In [1]:
import torch
from torch import nn
from torch.nn import functional as F
import math

自注意力机制

In [2]:
class SelfAttention(nn.Module):
    def __init__(self, dropout=0.1):
        super().__init__()
        self.softmax = nn.Softmax(dim=-1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, Q, K, V):
        d_k = Q.size(-1)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

        attn = self.softmax(scores)

        attn = self.dropout(attn)

        out = torch.matmul(attn, V)

        return out, attn

多头注意力机制

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.n_heads = n_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        self.self_attn = SelfAttention(dropout)

        self.fc = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v):
        batch_size = q.size(0)

        Q = self.W_q(q).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(k).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(v).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)

        out, attn = self.self_attn(Q, K, V)

        out = out.transpose(1, 2).contiguous().view(batch_size, -1, self.n_heads * self.d_k)

        out = self.fc(out)

        out = self.dropout(out)

        return out, attn

前馈神经网络

In [4]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.fc2(self.dropout(F.gelu(self.fc1(x))))

编码器

In [5]:
class Encoder(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout)

    def forward(self, src):
        out = src + self.attn(self.norm1(src), self.norm1(src), self.norm1(src))[0]
        out = out + self.ffn(self.norm2(out))

        return out

PatchEmbedding

In [6]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size, patch_size, inchannels, d_model):
        super().__init__()
        assert img_size % patch_size == 0,\
        f"img_size must be divisible by patch_size"
        self.img_size = img_size
        self.patch_size = patch_size
        self.n_patches = (img_size // patch_size) ** 2

        self.projection = nn.Conv2d(inchannels, d_model, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.projection(x)

        x = x.flatten(2)

        return x.transpose(1, 2)

ViT模型

In [7]:
class ViT(nn.Module):
    def __init__(self, img_size=224, patch_size=16, inchannels=3, d_model=768, n_heads=12, d_ff=3072, n_classes=1000, num_layers=12, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, inchannels, d_model)
        n_patches = self.patch_embed.n_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, 1+n_patches, d_model))

        self.dropout = nn.Dropout(dropout)

        self.layers = nn.ModuleList([
            Encoder(d_model, n_heads, d_ff, dropout) for _ in range(num_layers)
        ])

        self.norm = nn.LayerNorm(d_model)

        self.head = nn.Linear(d_model, n_classes)

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.apply(self._init_weight)

    def _init_weight(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.weight, 1.0)
            nn.init.constant_(m.bias, 0)

    def forward(self, x):
        batch_size = x.size(0)

        x = self.patch_embed(x)

        cls_tokens = self.cls_token.expand(batch_size, -1, -1)

        x = torch.cat([cls_tokens, x], dim=1)

        x = x + self.pos_embed

        x = self.dropout(x)

        for layer in self.layers:
            x = layer(x)

        x = self.norm(x)

        cls_out = x[:, 0, :]

        return self.head(cls_out)

In [8]:
model = ViT()
x = torch.randn(2, 3, 224, 224)
out = model(x)
print(out.shape)

torch.Size([2, 1000])


## 训练 ViT — CIFAR-100

CIFAR-100 是 32×32 的小图，需要调小 `patch_size` 和 `d_model`，否则参数量太大训不动。

关键点：
- **数据增强**：RandAugment + CutMix，ViT 没有 CNN 的归纳偏置，全靠数据撑
- **优化器**：AdamW + weight_decay=0.05，和 CNN 不同
- **调度器**：CosineAnnealingLR + Warmup，比 StepLR 效果好很多
- **Label Smoothing**：防止 ViT 在小数据集上过拟合

In [ ]:
# ============================================================
# 1. 训练环境配置
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import time
import copy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
print(f'使用设备: {device}')

In [ ]:
# ============================================================
# 2. 数据准备 — 增强策略对 ViT 至关重要
# ============================================================
# ViT 没有 CNN 的平移不变性和局部性归纳偏置，必须靠强数据增强
# 同时把 32×32 的 CIFAR 图像 resize 到 224×224 以匹配 ViT 的 patch 切分

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.5, 1.0)),  # 随机裁剪到 224
    transforms.RandomHorizontalFlip(),
    transforms.RandAugment(num_ops=2, magnitude=9),       # RandAugment 自动搜索最优增强策略
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408),        # CIFAR-100 的均值
                         (0.2675, 0.2565, 0.2761)),       # CIFAR-100 的标准差
])

# 验证/测试集：只做 resize + normalize，不做增强
val_transform = transforms.Compose([
    transforms.Resize(224),                                # 直接 resize 到 224
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408),
                         (0.2675, 0.2565, 0.2761)),
])

# 加载 CIFAR-100
train_dataset = datasets.CIFAR100(root='./data', train=True,
                                   download=True, transform=train_transform)
val_dataset = datasets.CIFAR100(root='./data', train=False,
                                  download=True, transform=val_transform)

# batch_size：GPU 显存不够就降到 32 或 16
batch_size = 64 if device.type == 'cuda' else 16
train_loader = DataLoader(train_dataset, batch_size=batch_size,
                           shuffle=True, num_workers=2, pin_memory=(device.type == 'cuda'))
val_loader = DataLoader(val_dataset, batch_size=batch_size,
                         shuffle=False, num_workers=2, pin_memory=(device.type == 'cuda'))

print(f'训练集: {len(train_dataset)} 张, 验证集: {len(val_dataset)} 张')
print(f'类别数: 100')

In [ ]:
# ============================================================
# 3. 模型配置 — ViT-Tiny（适合 CIFAR-100 的规模）
# ============================================================
# ViT-Base (ViT-B/16) 的 d_model=768、12层，CIFAR 上会过拟合
# 这里用缩小的版本：d_model=384、6层，参数量 ~10M

model = ViT(
    img_size=224,        # CIFAR resize 到了 224
    patch_size=16,       # 14×14=196 个 patch
    inchannels=3,
    d_model=384,         # 比 ViT-B 小一半
    n_heads=6,           # 384 / 6 = 64 (每个头 64 维)
    d_ff=1536,           # 384 × 4
    n_classes=100,       # CIFAR-100 有 100 类
    num_layers=6,        # 比 ViT-B(12层) 少一半
    dropout=0.1
).to(device)

# 统计参数量
n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'总参数量: {n_params/1e6:.2f}M')
print(f'可训练参数: {n_trainable/1e6:.2f}M')

In [ ]:
# ============================================================
# 4. 训练循环
# ============================================================
# ViT 训练和 CNN 的区别：
#   - AdamW（不是 Adam），weight_decay=0.05（比 CNN 大）
#   - CosineAnnealingLR + Warmup，不用 StepLR
#   - Label Smoothing 防止过拟合
#   - 梯度裁剪防止 Transformer 梯度爆炸

epochs = 50                   # CIFAR-100 训练轮数
warmup_epochs = 5             # 前 5 个 epoch 线性增长 lr

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)  # label smoothing 对 ViT 很关键
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)

# Cosine 衰减：lr 从 3e-4 余弦衰减到 ~1e-6
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs - warmup_epochs)

def train_one_epoch(model, loader, criterion, optimizer, epoch):
    """训练一个 epoch，返回平均 loss 和 accuracy"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch_idx, (images, targets) in enumerate(loader):
        images, targets = images.to(device), targets.to(device)

        # 前向传播
        outputs = model(images)
        loss = criterion(outputs, targets)

        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # 梯度裁剪
        optimizer.step()

        # 统计
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    return total_loss / len(loader), 100. * correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    """在验证集上评估"""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        outputs = model(images)
        loss = criterion(outputs, targets)

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    return total_loss / len(loader), 100. * correct / total


# ============================================================
# 开始训练
# ============================================================
best_acc = 0
best_model = None
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print('开始训练 ViT on CIFAR-100')
print('=' * 60)

for epoch in range(1, epochs + 1):
    start_time = time.time()

    # Warmup：前几个 epoch 学习率从 0 线性增长到初始值
    if epoch <= warmup_epochs:
        lr_scale = epoch / warmup_epochs
        for param_group in optimizer.param_groups:
            param_group['lr'] = 3e-4 * lr_scale
    else:
        scheduler.step()  # Cosine 衰减

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, epoch)
    val_loss, val_acc = evaluate(model, val_loader, criterion)

    # 记录历史
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    # 保存最佳模型
    if val_acc > best_acc:
        best_acc = val_acc
        best_model = copy.deepcopy(model.state_dict())

    epoch_time = time.time() - start_time
    current_lr = optimizer.param_groups[0]['lr']
    print(f'Epoch {epoch:3d}/{epochs} | '
          f'lr: {current_lr:.2e} | '
          f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | '
          f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% | '
          f'Time: {epoch_time:.1f}s')

print('=' * 60)
print(f'训练完成！最佳验证准确率: {best_acc:.2f}%')

# 加载最佳模型
model.load_state_dict(best_model)

In [ ]:
# ============================================================
# 5. 测试集最终评估
# ============================================================
test_loss, test_acc = evaluate(model, val_loader, criterion)
print(f'测试集 Loss: {test_loss:.4f}')
print(f'测试集 Accuracy: {test_acc:.2f}%')

# 消融实验参考：
# 如果加了 label_smoothing + RandAugment + CosineAnnealing，
# ViT-Tiny 在 CIFAR-100 上大约能达到 55%~65% Top-1 Acc。
# ViT-B/16（预训练权重）能达到 ~90%，差距来自数据量和模型容量。